In [1]:
import base64
import requests
import pandas as pd
from time import sleep
from tqdm import tqdm

# Your Spotify API credentials
CLIENT_ID = "3b20bf9f77ba4be9b694c3c8a702a13d"
CLIENT_SECRET = "c9dc00b3db684750b0412760fce203c6"

In [ ]:
# ---- Step 1: Get access token automatically ----
def get_access_token(client_id, client_secret):
    token_url = "https://accounts.spotify.com/api/token"
    credentials = f"{client_id}:{client_secret}"
    encoded = base64.b64encode(credentials.encode()).decode()
    headers = {"Authorization": f"Basic {encoded}"}
    data = {"grant_type": "client_credentials"}
    r = requests.post(token_url, headers=headers, data=data, timeout=10)
    r.raise_for_status()
    return r.json()["access_token"]

In [3]:
# Get token
ACCESS_TOKEN = get_access_token(CLIENT_ID, CLIENT_SECRET)
HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

In [6]:
df = pd.read_csv("data/features-annotated-artistless-with-lyrics-and-languages.csv")

# ---- Step 3: Add new columns for fetched data ----
df["followers"] = None
df["genres"] = None
df["popularity"] = None
# Add new columns
df["artist_id"] = None
df["explicit"] = None
df["track_popularity"] = None

In [ ]:
def get_tracks_batch(track_ids, retries=3, delay=2):
    url = "https://api.spotify.com/v1/tracks"
    params = {"ids": ",".join(track_ids)}

    for attempt in range(retries):
        try:
            response = requests.get(url, headers=HEADERS, params=params, timeout=20)
            if response.status_code == 200:
                return response.json()["tracks"]
            elif response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", 5))
                print(f"⏳ Rate limit hit. Waiting {retry_after}s...")
                sleep(retry_after)
            else:
                print(f"⚠️ Error {response.status_code}: {response.text[:100]}")
                sleep(delay)
        except requests.exceptions.Timeout:
            print(f"⏳ Timeout, retrying batch ({attempt+1}/{retries})...")
            sleep(delay)
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Network error: {e}")
            sleep(delay)
    return []  # failed batch

In [12]:
import random
batch_size = 50
ids = df["id"].dropna().astype(str).tolist()

for start in tqdm(range(0, len(ids), batch_size)):
    batch_ids = ids[start:start + batch_size]
    tracks = get_tracks_batch(batch_ids)
    
    for track in tracks:
        if not track:
            print(f"⚠️ Failed batch starting at index {start}")
        track_id = track["id"]
        artist_id = track["artists"][0]["id"] if track["artists"] else None
        explicit = track.get("explicit", None)
        popularity = track.get("popularity", None)

        df.loc[df["id"] == track_id, ["artist_id", "explicit", "track_popularity"]] = [artist_id, explicit, popularity]

sleep(random.uniform(0.2, 0.6))


 ... (more hidden) ...


In [13]:
output_path = "data/additional_features.csv"
df.to_csv(output_path, index=False)
print(f"✅ Done! Saved enriched data to {output_path}")

✅ Done! Saved enriched data to data/additional_features.csv


In [15]:
#print the df's artist_id column and explicit column and track_popularity column to verify they have been added
print(df[["artist_id", "explicit", "track_popularity"]].head(10))

                artist_id explicit track_popularity
0  4q3ewBCX7sLwd24euuV69X     True               78
1  66CXWjxzNUsdJxJ2JdwvnR     True                0
2  0XeEobZplHxzM9QzFQWLiR    False               73
3  246dkjvS1zLTtiykXe5h60    False                0
4  7n2wHs1TKAczGzO7Dd2rGr    False               79
5  1Xyo4u8uXC1ZmMpatF05PJ    False               40
6  4dpARuHxo51G3z768sgnrY    False               78
7  1McMsnEElThX1knmY4oliG     True               21
8  0iEtIxbK0KxaSlF7G42ZOp     True                1
9  06HL4z0CvFAxyc27GXpf02    False                0


In [23]:
def get_access_token2(client_id, client_secret):
    token_url = "https://accounts.spotify.com/api/token"
    credentials = f"{client_id}:{client_secret}"
    encoded = base64.b64encode(credentials.encode()).decode()
    headers = {"Authorization": f"Basic {encoded}"}
    data = {"grant_type": "client_credentials"}
    r = requests.post(token_url, headers=headers, data=data, timeout=10)
    r.raise_for_status()
    return r.json()["access_token"]

ACCESS_TOKEN2 = get_access_token2(CLIENT_ID, CLIENT_SECRET)
HEADERS2 = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

In [24]:
def get_artists_batch(artist_ids, retries=3, delay=2):
    url = "https://api.spotify.com/v1/artists"
    params = {"ids": ",".join(artist_ids)}

    for attempt in range(retries):
        try:
            response = requests.get(url, headers=HEADERS, params=params, timeout=10)

            if response.status_code == 200:
                return response.json()["artists"]
            elif response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", 5))
                print(f"⏳ Rate limited, waiting {retry_after}s...")
                sleep(retry_after)
            elif response.status_code == 401:
                # Token expired, refresh it
                global ACCESS_TOKEN2, HEADERS2
                print("🔄 Token expired. Refreshing...")
                ACCESS_TOKEN2 = get_access_token(CLIENT_ID, CLIENT_SECRET)
                HEADERS2 = {"Authorization": f"Bearer {ACCESS_TOKEN2}"}
            else:
                print(f"⚠️ Error {response.status_code}: {response.text[:100]}")
                sleep(delay)
        except requests.exceptions.Timeout:
            print(f"⏳ Timeout, retrying batch ({attempt+1}/{retries})...")
            sleep(delay)
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Network error: {e}")
            sleep(delay)
    return []  # failed batch

In [25]:
artist_ids = df["artist_id"].dropna().astype(str).unique().tolist()
batch_size = 50

for start in tqdm(range(0, len(artist_ids), batch_size)):
    batch = artist_ids[start:start + batch_size]
    artists = get_artists_batch(batch)

    for artist in artists:
        if not artist:
            continue
        aid = artist["id"]
        followers = artist["followers"]["total"]
        genres = ", ".join(artist["genres"]) if artist["genres"] else None
        popularity = artist["popularity"]

        df.loc[df["artist_id"] == aid, ["artist_followers", "artist_genres", "artist_popularity"]] = [
            followers, genres, popularity
        ]

    # polite delay (with jitter)
    sleep(random.uniform(0.2, 0.6))

 ... (more hidden) ...


In [26]:
# ---- Step 5: Save enriched dataset ----
output_path = "data/additional_features.csv"
df.to_csv(output_path, index=False)
print(f"✅ Done! Saved enriched data to {output_path}")

✅ Done! Saved enriched data to data/additional_features.csv


In [27]:
#print the new columns to verify they have been added
print(df[["artist_id", "artist_followers", "artist_genres", "artist_popularity"]].head(10))

                artist_id  artist_followers  \
0  4q3ewBCX7sLwd24euuV69X       102185553.0   
1  66CXWjxzNUsdJxJ2JdwvnR       107261237.0   
2  0XeEobZplHxzM9QzFQWLiR         6434994.0   
3  246dkjvS1zLTtiykXe5h60        47686394.0   
4  7n2wHs1TKAczGzO7Dd2rGr        45665812.0   
5  1Xyo4u8uXC1ZmMpatF05PJ       112582198.0   
6  4dpARuHxo51G3z768sgnrY        66572253.0   
7  1McMsnEElThX1knmY4oliG        48901491.0   
8  0iEtIxbK0KxaSlF7G42ZOp        11643641.0   
9  06HL4z0CvFAxyc27GXpf02       145000314.0   

                                       artist_genres  artist_popularity  
0       reggaeton, trap latino, urbano latino, latin               95.0  
1                                                pop               90.0  
2  corridos tumbados, sad sierreño, corrido, sier...               75.0  
3                                               None               85.0  
4                                               None               81.0  
5                                     